# F1 Race Strategy - explorationScratch notebook used while building the pipeline. It reads the artifacts in `data/`;it does **not** define any pipeline logic. Everything authoritative lives in `src/`.Run order: `src/ingest_resumable.py` -> `src/features.py` -> `src/degradation.py` -> `src/models.py`.

In [ ]:
import sys, jsonsys.path.insert(0, "../src")import pandas as pd, numpy as npimport matplotlib.pyplot as pltfrom config import DATA_DIRdf = pd.read_parquet(DATA_DIR / "features.parquet")print(df.shape, sorted(df.year.unique()))df.head()

## 1. What cleaning removedThe attrition table is the evidence that the filtering was inspected rather than assumed.

In [ ]:
pd.read_csv(DATA_DIR / "lap_attrition.csv")

## 2. Does fuel correction do what it should?Raw lap times fall through a race as the car burns fuel. After correcting to anequivalent empty-tank time, that downward drift should be gone (what remains istrack evolution and tyre state). Corrected times are *slower* than raw byconstruction - we are adding back the time the fuel load cost.

In [ ]:
g = df.groupby("lap_number")[["lap_time", "lap_time_fuel_corrected"]].median()ax = g.plot(figsize=(9,4))ax.set_xlabel("Lap number"); ax.set_ylabel("Field median lap time (s)")ax.set_title("Raw vs fuel-corrected pace over a race (pooled across races)")plt.show()print("mean fuel correction applied (s):",      (df.lap_time_fuel_corrected - df.lap_time).mean().round(3))print("correlation of raw pace with lap number     :", df.lap_time.corr(df.lap_number).round(3))print("correlation of corrected pace with lap number:", df.lap_time_fuel_corrected.corr(df.lap_number).round(3))

## 3. Track evolutionPer race, the field's median fuel-corrected lap time by lap number, smoothed.Positive `track_evolution_gain_s` means the track has rubbered in and is fasterthan it was at the start.

In [ ]:
ev = (df.groupby(["year","round","lap_number"])["track_evolution_gain_s"].first()        .reset_index())fig, ax = plt.subplots(figsize=(9,4))for (y,r), g in ev.groupby(["year","round"]):    ax.plot(g.lap_number, g.track_evolution_gain_s, alpha=.25, lw=.8)ax.plot(ev.groupby("lap_number").track_evolution_gain_s.median(), color="k", lw=2.5,        label="median across races")ax.set_xlabel("Lap"); ax.set_ylabel("Pace gain vs lap 1 (s)"); ax.legend()ax.set_title("Track evolution, one line per race"); plt.show()

## 4. Degradation slopesThe headline result. SOFT must degrade fastest; if it does not, something upstream is broken.

In [ ]:
slopes = pd.read_csv(DATA_DIR / "degradation_slopes.csv")display(slopes)circ = DATA_DIR / "degradation_slopes_by_circuit.csv"if circ.exists():    c = pd.read_csv(circ)    piv = c.pivot_table(index="circuit", columns="compound", values="deg_s_per_lap")    display(piv.reindex(columns=["SOFT","MEDIUM","HARD"]).sort_values("SOFT", ascending=False).round(4))

In [ ]:
from IPython.display import ImageImage(filename="../figures/degradation_curves.png")

## 5. Pit-window labelRare-event problem. The base rate is the number that makes accuracy meaninglesshere, so it is worth looking at directly.

In [ ]:
print("base rate overall: %.2f%%" % (df.pits_within_horizon.mean()*100))print(df.groupby("year").pits_within_horizon.mean().round(4))ax = df.groupby("tyre_life").pits_within_horizon.mean().head(45).plot(figsize=(9,4))ax.set_xlabel("Tyre age (laps)"); ax.set_ylabel("P(pit within 3 laps)")ax.set_title("Empirical pit hazard by tyre age"); plt.show()

## 6. Model metrics and the leakage checklist

In [ ]:
print(json.dumps(json.load(open(DATA_DIR / "model_metrics.json")), indent=2))

In [ ]:
lc = pd.read_csv(DATA_DIR / "leakage_checklist.csv")for _, r in lc.iterrows():    print(f"[{r['status']}] {r['item']}\n    {r['detail'][:300]}\n")